# Bab 7 — Calinski-Harabasz Index

Notebook pendamping Bab 7 Level 5.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Menjelaskan ide *Calinski-Harabasz Index* (CHI), juga disebut *Variance Ratio Criterion*.
- Menjelaskan dispersi antar kelompok (*between-cluster dispersion*) dan dispersi dalam kelompok (*within-cluster dispersion*).
- Menghitung CHI secara manual dan mencocokkannya dengan `calinski_harabasz_score`.
- Menafsirkan nilai CHI dan menjelaskan mengapa **lebih tinggi lebih baik**.
- Menjelaskan bahwa nilai CHI dipengaruhi jumlah data dan jumlah kelompok.
- Menyebutkan kelebihan dan keterbatasan CHI, termasuk bias terhadap kelompok bulat.
- Membandingkan Silhouette, Davies-Bouldin, dan Calinski-Harabasz pada data yang sama, serta memahami mengapa ketiganya bisa tidak sepakat.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

## Bagian B: Sembilan titik dan tiga kelompok

In [ ]:
X_tiny = np.array(
    [
        [1, 1], [2, 1], [1, 2],   # kelompok 0
        [6, 5], [7, 5], [6, 6],   # kelompok 1
        [1, 8], [2, 8], [1, 9],   # kelompok 2
    ],
    dtype=float,
)
labels_tiny = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])

## Bagian C: CH dihitung manual

In [ ]:
def ch_manual(X, labels, verbose=False):
    """Calinski-Harabasz Index dihitung langkah demi langkah."""
    n = len(X)
    ks = np.unique(labels)
    k = len(ks)
    overall_centroid = X.mean(axis=0)

    B = 0.0   # dispersi ANTAR kelompok
    W = 0.0   # dispersi DALAM kelompok
    for c in ks:
        members = X[labels == c]
        centroid = members.mean(axis=0)
        B += len(members) * ((centroid - overall_centroid) ** 2).sum()
        W += ((members - centroid) ** 2).sum()

    ch = (B / (k - 1)) / (W / (n - k))
    if verbose:
        total = ((X - overall_centroid) ** 2).sum()
        print("Centroid keseluruhan :", overall_centroid.round(3).tolist())
        print(f"B (antar kelompok)   : {B:.3f}")
        print(f"W (dalam kelompok)   : {W:.3f}")
        print(f"B + W                : {B + W:.3f}   (jumlah kuadrat total = {total:.3f})")
        print(f"n = {n}, k = {k}  ->  (B / {k - 1}) / (W / {n - k})")
    return ch


ch_tiny = ch_manual(X_tiny, labels_tiny, verbose=True)
print(f"CH manual = {ch_tiny:.4f}")

## Bagian D: Mencocokkan dengan scikit-learn

In [ ]:
print(f"calinski_harabasz_score = {calinski_harabasz_score(X_tiny, labels_tiny):.4f}")

## Bagian E: CH untuk beberapa nilai K

In [ ]:
X_blobs, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.0, random_state=42)
rows = []
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_blobs)
    rows.append({"K": k, "Calinski-Harabasz": calinski_harabasz_score(X_blobs, labels)})
blob_table = pd.DataFrame(rows).set_index("K")
print(blob_table.round(1).to_string())
print("K dengan CH tertinggi:", int(blob_table["Calinski-Harabasz"].idxmax()))

## Bagian F: Skala dan jumlah data

In [ ]:
labels4 = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_blobs)
print(f"CH data asli          : {calinski_harabasz_score(X_blobs, labels4):.2f}")
print(f"CH data x 10          : {calinski_harabasz_score(X_blobs * 10, labels4):.2f}")

# Data digandakan: bentuk kelompok sama, jumlah titik dua kali lipat
X_double = np.vstack([X_blobs, X_blobs])
labels_double = np.concatenate([labels4, labels4])
print(f"CH data digandakan    : {calinski_harabasz_score(X_double, labels_double):.2f}   (400 -> 800 titik)")

## Bagian G: Data tanpa kelompok alami

In [ ]:
X_uniform = np.random.default_rng(42).uniform(-10, 10, size=(400, 2))
uniform_ch = {}
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_uniform)
    uniform_ch[k] = calinski_harabasz_score(X_uniform, labels)
print({k: round(v, 1) for k, v in uniform_ch.items()})
print("K dengan CH tertinggi pada data acak:", max(uniform_ch, key=uniform_ch.get))

## Bagian H: Keterbatasan pada moons

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.06, random_state=42)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)
print(f"CH K-Means (memotong bulan) : {calinski_harabasz_score(X_moons, km_moons):.1f}")
print(f"CH label benar              : {calinski_harabasz_score(X_moons, y_moons):.1f}")

## Bagian I: Tiga metrik internal bersama-sama

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_mall = StandardScaler().fit_transform(df[features])

rows = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_mall)
    rows.append(
        {
            "K": k,
            "Silhouette": silhouette_score(X_mall, labels),
            "Davies-Bouldin": davies_bouldin_score(X_mall, labels),
            "Calinski-Harabasz": calinski_harabasz_score(X_mall, labels),
        }
    )
mall_table = pd.DataFrame(rows).set_index("K")
print(mall_table.round(3).to_string())
print("K terbaik menurut Silhouette (tertinggi)        :", int(mall_table["Silhouette"].idxmax()))
print("K terbaik menurut Davies-Bouldin (terendah)     :", int(mall_table["Davies-Bouldin"].idxmin()))
print("K terbaik menurut Calinski-Harabasz (tertinggi) :", int(mall_table["Calinski-Harabasz"].idxmax()))

## Bagian J: Menggambar CH terhadap K

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(blob_table.index, blob_table["Calinski-Harabasz"], marker="o", color=palette[0])
axes[0].set_title("Empat gerombol jelas")
axes[1].plot(list(uniform_ch.keys()), list(uniform_ch.values()), marker="o", color=palette[1])
axes[1].set_title("Tanpa kelompok alami")
axes[2].plot(mall_table.index, mall_table["Calinski-Harabasz"], marker="o", color=palette[2])
axes[2].set_title("Mall Customers")
for ax in axes:
    ax.set_xlabel("Jumlah kelompok (K)")
    ax.set_ylabel("Calinski-Harabasz Index (lebih tinggi lebih baik)")
plt.tight_layout()
plt.show()

## Bagian K: Menggambar tiga metrik pada Mall Customers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, column, color in zip(axes, mall_table.columns, palette):
    ax.plot(mall_table.index, mall_table[column], marker="o", color=color)
    best = mall_table[column].idxmin() if column == "Davies-Bouldin" else mall_table[column].idxmax()
    ax.axvline(best, color="gray", linestyle="--")
    ax.set_title(f"{column}\n(terbaik: K = {best})")
    ax.set_xlabel("Jumlah kelompok (K)")
plt.tight_layout()
plt.show()

## Latihan

1. Hitung manual \( B_k \) untuk kelompok 1 pada `X_tiny` dan cocokkan dengan 33,667.
2. Geser titik \( (7,5) \) menjadi \( (9,5) \). Berapa CHI-nya? Mengapa turun?
3. Hitung CHI `X_tiny` untuk K-Means dengan K = 2 dan K = 4, dan bandingkan dengan K = 3 (93,0).
4. Ganti `make_blobs` pada `X_blobs` menjadi `n_samples=800`. Apakah CHI K = 4 naik? Mengapa?
5. Pada Mall Customers, hitung selisih persen CHI antara K = 5 dan K = 9. Apakah selisih itu cukup besar untuk memilih K = 9?
6. Mengapa CHI pada data acak bisa tertinggi di K terbesar?
7. Mengapa kita tidak boleh membandingkan CHI antar dataset dengan jumlah titik berbeda?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Calinski-Harabasz Index** (*Variance Ratio Criterion*) adalah rasio dispersi antar kelompok terhadap dispersi dalam kelompok, dikoreksi dengan derajat kebebasan: \( \text{CH} = \dfrac{B_k/(k-1)}{W_k/(n-k)} \).
- \( W_k \) adalah WCSS (Bab 3), dan \( B_k + W_k \) sama dengan dispersi total. Makin **tinggi** CHI, makin baik. Tidak ada batas atas dan ambang baku.
- Hitungan manual pada sembilan titik menghasilkan **93,0**, sama dengan `calinski_harabasz_score`.
- Pada empat gerombol jelas, CHI memuncak tegas di K = 4 (4580,4).
- CHI tidak bergantung pada skala keseluruhan (data × 10 tetap 4580,38), tetapi **bergantung pada jumlah data** (data digandakan memberi 9207,02, sekitar 2,01 kali).
- CHI bisa terus naik pada data tanpa struktur (tertinggi di K = 10 pada data acak) dan menunjuk K = 9 pada Mall Customers, sementara Silhouette dan Davies-Bouldin menunjuk K = 5.
- Seperti metrik internal lain, CHI menyukai kelompok bulat (pada moons, label benar 198,3 vs K-Means 440,2).
- Jangan menafsirkan CHI sendirian. Padukan beberapa metrik, gambar, dan pengetahuan domain.